# Lab 19: Tree-Based Models — Random Forests
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 30 min Core + 15 min Extension + SHAP Deep Dive

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

---

In [1]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load data
# -----------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## Part 1: Find the Bug — Model Comparison (10 min)

The following code trains three models and reports their performance.
**Something is wrong with how the comparison is set up.** Find it, fix it, explain.

In [2]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains deliberate error)
# Step 2: Model comparison — find the bug
# -----------------------------------------------------------

tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# BUG IS HERE: RF is evaluated on TRAINING data, not test data
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print('=== Model Comparison ===')
print(f"Single Tree  \u2014 R\u00b2: {r2_score(y_test, tree.predict(X_test)):.4f}")
print(f"Ridge        \u2014 R\u00b2: {r2_score(y_test, ridge.predict(X_test)):.4f}")
print(f"Random Forest \u2014 R\u00b2: {r2_score(y_train, rf.predict(X_train)):.4f}")  # \u2190 WRONG: using training set
print()
print('Conclusion: Random Forest achieves R\u00b2 > 0.97! Far superior to alternatives.')

=== Model Comparison ===
Single Tree  — R²: 0.6187
Ridge        — R²: 0.5759
Random Forest — R²: 0.9735

Conclusion: Random Forest achieves R² > 0.97! Far superior to alternatives.


### YOUR DIAGNOSIS

1. **What is wrong?** (identify the specific line and error type)
2. **Why is this dangerous?** (what misleading conclusion does it lead to?)
3. **Fix the code below** and report the correct R²

**Verification checkpoint:** After fixing, the RF Test R² should be between 0.78 and 0.83. If you get >0.95, you haven't found the bug.

4. **Which chapter concept does this error violate?** (hint: Ch 15)

In [3]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Fix the model comparison bug from Part 1
# -----------------------------------------------------------
tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print('=== Model Comparison ===')
print(f"Single Tree  \u2014 R\u00b2: {r2_score(y_test, tree.predict(X_test)):.4f}")
print(f"Ridge        \u2014 R\u00b2: {r2_score(y_test, ridge.predict(X_test)):.4f}")
print(f"Random Forest \u2014 R\u00b2: {r2_score(y_test, rf.predict(X_test)):.4f}")
print()
print('Previous code uses the training data instead of the test data. RF models overfit training data so the results have artificially inflated correlated')
print(f"The correct R\u00b2: {r2_score(y_test, rf.predict(X_test)):.4f}")


=== Model Comparison ===
Single Tree  — R²: 0.6187
Ridge        — R²: 0.5759
Random Forest — R²: 0.8049

Previous code uses the training data instead of the test data. RF models overfit training data so the results have artificially inflated correlated
The correct R²: 0.8049


## Part 2: Find the Methodological Flaw — Feature Importance (10 min)

The following analysis uses feature importance to make a **causal claim**.
The code runs correctly. The methodology is wrong. Find the flaw.

In [4]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains methodological flaw)
# Step 3: Feature importance with flawed causal reasoning
# -----------------------------------------------------------

rf_correct = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf_correct.fit(X_train, y_train)

importance = pd.Series(rf_correct.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Feature Importance (MDI):')
print(importance.round(4))
print()
print('POLICY RECOMMENDATION:')
print(f'The top predictor is {importance.index[0]} (importance = {importance.iloc[0]:.3f}).')
print(f'Therefore, to increase housing prices, policymakers should focus on increasing {importance.index[0]}.')
print(f'The second most important lever is {importance.index[1]}.')

Feature Importance (MDI):
MedInc        0.5259
AveOccup      0.1381
Latitude      0.0886
Longitude     0.0883
HouseAge      0.0543
AveRooms      0.0444
Population    0.0306
AveBedrms     0.0297
dtype: float64

POLICY RECOMMENDATION:
The top predictor is MedInc (importance = 0.526).
Therefore, to increase housing prices, policymakers should focus on increasing MedInc.
The second most important lever is AveOccup.


### YOUR DIAGNOSIS

1. **What is the methodological flaw?** (the code is correct — the reasoning is wrong)
2. **Why can't we use MDI for policy recommendations?** (connect to Ch 10 DAGs and Ch 15 prediction vs. explanation)
3. **What would you need to make a causal claim?** (hint: Ch 24 DML)
4. **Bonus:** MDI has a known statistical bias. What is it, and what alternative would you use?

**Verification checkpoint:** Your diagnosis should mention at least: (a) prediction ≠ causation, (b) confounding/omitted variables, (c) MDI bias toward high-cardinality features.

In [5]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Run permutation importance and write a proper (non-causal)
# interpretation of the results
# -----------------------------------------------------------
rf_correct = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf_correct.fit(X_train, y_train)

#importance = pd.Series(rf_correct.feature_importances_, index=X.columns).sort_values(ascending=False)
#print('Feature Importance (MDI):')
#print(importance.round(4))
#print()
result = permutation_importance(rf_correct, X_test,y_test, n_repeats=10,random_state=RANDOM_STATE)
perm_imp = pd.Series(result.importances_mean, index=X.columns).sort_values(ascending=False)
print("Permutation Importance")
print(perm_imp.round(4))

print('POLICY RECOMMENDATION:')
print(f'The top predictor is {perm_imp.index[0]} (perm_imp = {perm_imp.iloc[0]:.3f}).')
print(f'{perm_imp.index[0]} is the strongest statistical indicator of price in the dataset, however because prediction cannot be a substitute for causation, \nand the presence of ommitted variables may be influencing the predictive ability of the variable, further analysis would be needed in order to prove a strong causal relationship')
print(f'The second most important lever is {perm_imp.index[1]}.')

Permutation Importance
MedInc        0.7299
Latitude      0.4444
Longitude     0.3352
AveOccup      0.2029
HouseAge      0.0722
AveRooms      0.0274
AveBedrms     0.0096
Population    0.0078
dtype: float64
POLICY RECOMMENDATION:
The top predictor is MedInc (perm_imp = 0.730).
MedInc is the strongest statistical indicator of price in the dataset, however because prediction cannot be a substitute for causation, 
and the presence of ommitted variables may be influencing the predictive ability of the variable, further analysis would be needed in order to prove a strong causal relationship
The second most important lever is Latitude.


## Part 3: Hyperparameter Tuning + XGBoost Comparison (10 min)

Tune the RF, then compare against XGBoost (gradient boosting).

In [6]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Tune RF with GridSearchCV and compare with GBR
# -----------------------------------------------------------

print("1. GridSearchCV on RandomForestRegressor")
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 0.5],
}

param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 0.5],
}

rf_default = RandomForestRegressor(random_state=RANDOM_STATE)
rf_default.fit(X_train, y_train)

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
rf_tuned = grid_search.best_estimator_
print("Best RF params:", grid_search.best_params_)

print("2. Fit GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1)")
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=5, 
                                 learning_rate=0.1, random_state=RANDOM_STATE)
gbr.fit(X_train, y_train)

print("3. Compare Test RMSE and R\u00b2 for: Ridge, RF (default), RF (tuned), GBR")
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

models = {
    'Ridge':     ridge,
    'RF Default': rf_default,
    'RF Tuned':  rf_tuned,
    'GBR':       gbr,
}

print(f"\n{'Model':<12} {'RMSE':>10} {'R²':>10}")
print("-" * 34)
results = {}
for name, model in models.items():
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'rmse': rmse, 'r2': r2}
    print(f"{name:<12} {rmse:>10.4f} {r2:>10.4f}")

print("4. Which model wins? By how much? Is the difference practically significant?")
winner = min(results, key=lambda m: results[m]['rmse'])
baseline_rmse = results['Ridge']['rmse']
winner_rmse   = results[winner]['rmse']
improvement   = (baseline_rmse - winner_rmse) / baseline_rmse * 100

print(f"\nWinner: {winner}")
print(f"RMSE improvement over Ridge: {improvement:.1f}%")

print("is the gap meaningful?")
gbr_rmse = results['GBR']['rmse']
tuned_rmse = results['RF Tuned']['rmse']
marginal_diff = abs(gbr_rmse - tuned_rmse) / baseline_rmse * 100
print(f"Marginal difference GBR vs RF Tuned: {marginal_diff:.1f}% of baseline RMSE")
if marginal_diff < 2:
    print("→ Practically insignificant. Prefer simpler model (RF Tuned) for interpretability.")
else:
    print("→ Meaningful difference. GBR complexity is justified.")

1. GridSearchCV on RandomForestRegressor
Best RF params: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 500}
2. Fit GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1)
3. Compare Test RMSE and R² for: Ridge, RF (default), RF (tuned), GBR

Model              RMSE         R²
----------------------------------
Ridge            0.7455     0.5759
RF Default       0.5057     0.8049
RF Tuned         0.4928     0.8147
GBR              0.4736     0.8288
4. Which model wins? By how much? Is the difference practically significant?

Winner: GBR
RMSE improvement over Ridge: 36.5%
is the gap meaningful?
Marginal difference GBR vs RF Tuned: 2.6% of baseline RMSE
→ Meaningful difference. GBR complexity is justified.


---

## Extension: SHAP Analysis (5200 depth — 15 min)

Use SHAP to explain individual predictions. Compare MDI ranking vs. SHAP ranking.

In [7]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 4: SHAP setup and TreeExplainer
# -----------------------------------------------------------

# Install SHAP if needed
!pip install shap
import shap

# Recompute everything fresh - don't reuse old shap_values
X_sample = X_test.sample(200, random_state=42).reset_index(drop=True)
shap_values = explainer.shap_values(X_sample, approximate=True)  # must rerun this

explanation = shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=X_sample.values,                    # ← .values converts to numpy, bypasses index entirely
    feature_names=X_sample.columns.tolist()
)

shap.plots.waterfall(explanation[0])
shap.plots.beeswarm(explanation)

NameError: name 'explainer' is not defined

### SHAP Interpretation (write as a .py module)

Create a reusable `shap_analysis.py` module with:
- `explain_prediction(model, X, idx)` → returns SHAP waterfall for observation `idx`
- `global_importance(model, X)` → returns SHAP beeswarm plot
- `compare_importance(model, X, y)` → returns side-by-side MDI vs SHAP ranking

Include docstrings and type hints. This is a portfolio artifact.

# shap_analysis.py

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor


def explain_prediction(model, X: pd.DataFrame, idx: int, sample_size: int = 200) -> None:
    """
    Display a SHAP waterfall plot for a single observation.

    Parameters
    ----------
    model       : fitted sklearn tree-based model
    X           : feature DataFrame (will be sampled if large)
    idx         : positional index of the observation to explain
    sample_size : max rows to compute SHAP values over
    """
    X_sample = X.sample(min(sample_size, len(X)), random_state=42).reset_index(drop=True)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample.values, approximate=True)

    explanation = shap.Explanation(
        values=shap_values,
        base_values=explainer.expected_value,
        data=X_sample.values,
        feature_names=X.columns.tolist()
    )
    shap.plots.waterfall(explanation[idx])


def global_importance(model, X: pd.DataFrame, sample_size: int = 200) -> None:
    """
    Display a SHAP beeswarm plot showing global feature importance.

    Parameters
    ----------
    model       : fitted sklearn tree-based model
    X           : feature DataFrame
    sample_size : max rows to compute SHAP values over
    """
    X_sample = X.sample(min(sample_size, len(X)), random_state=42).reset_index(drop=True)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample.values, approximate=True)

    explanation = shap.Explanation(
        values=shap_values,
        base_values=explainer.expected_value,
        data=X_sample.values,
        feature_names=X.columns.tolist()
    )
    shap.plots.beeswarm(explanation)


def compare_importance(model, X: pd.DataFrame, sample_size: int = 200) -> pd.DataFrame:
    """
    Compare MDI vs SHAP feature importance rankings side by side.

    Parameters
    ----------
    model       : fitted sklearn RandomForest (must have feature_importances_)
    X           : feature DataFrame
    sample_size : max rows to compute SHAP values over

    Returns
    -------
    pd.DataFrame with MDI rank, SHAP rank, and both importance scores
    """
    X_sample = X.sample(min(sample_size, len(X)), random_state=42).reset_index(drop=True)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample.values, approximate=True)

    mdi = pd.Series(model.feature_importances_, index=X.columns, name="MDI")
    shap_imp = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns, name="SHAP")

    comparison = pd.DataFrame({"MDI": mdi, "SHAP": shap_imp})
    comparison["MDI_rank"]  = comparison["MDI"].rank(ascending=False).astype(int)
    comparison["SHAP_rank"] = comparison["SHAP"].rank(ascending=False).astype(int)
    comparison["rank_diff"] = (comparison["MDI_rank"] - comparison["SHAP_rank"]).abs()
    comparison = comparison.sort_values("SHAP_rank")

    print(comparison[["MDI_rank", "SHAP_rank", "rank_diff", "MDI", "SHAP"]].round(4))
    return comparison

---
## AI-Assisted Expansion: SHAP Dashboard + Reusable Module

**The Generative AI Policy: Foundations First, Expansion Second.** You have now established manual mastery over decision trees, random forests, hyperparameter tuning, feature importance, and SHAP explanations. You are now authorized to operate under the "Co-Pilot Rule."

### Your Expansion Task (5200 — Advanced)
Build TWO artifacts:

**Artifact 1: `src/shap_utils.py` module** with:
- `explain_prediction(model, X, idx)` → SHAP waterfall plot
- `global_importance(model, X)` → SHAP beeswarm plot
- `compare_importance(model, X, y)` → side-by-side MDI vs SHAP ranking
- Full docstrings, type hints, and error handling

**Artifact 2: Interactive Streamlit app** that lets the user:
1. Adjust `n_estimators` (1-500) and `max_features` (1-8) with sliders
2. See SHAP waterfall + beeswarm plots update with each parameter change
3. Compare RF vs Ridge vs GBR performance as hyperparameters change
4. Toggle between MDI, permutation, and SHAP importance rankings

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [ ]:
import ipywidgets as widgets
widgets.IntSlider(value=5, min=0, max=10)

IntSlider(value=5, max=10)

In [ ]:
pip install plotly nbformat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 15.7 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]
Note: you may need to restart the kernel to use updated packages.


In [8]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Co-Pilot required
# Copy the P.R.I.M.E. prompt above into Claude, then paste
# the generated code here. Run it and verify.
# -----------------------------------------------------------

# [Prep] Act as an expert Python Data Scientist specializing
# in SHAP explanations, interactive visualizations, and
# scikit-learn production workflows.
#
# [Request] I just completed a diagnosis-first lab where I
# compared Decision Trees, Ridge, Random Forests, and Gradient
# Boosting on California Housing data. I fixed evaluation bugs,
# diagnosed causal overclaiming from MDI, tuned hyperparameters
# with GridSearchCV, and generated SHAP waterfall + beeswarm
# plots. Now I need TWO artifacts:
#
# 1. A reusable `src/shap_utils.py` module with three functions:
#    - explain_prediction(model, X, idx) -> SHAP waterfall
#    - global_importance(model, X) -> SHAP beeswarm
#    - compare_importance(model, X, y) -> MDI vs SHAP side-by-side
#    Include type hints, docstrings, and error handling.
#
# 2. An interactive Plotly dashboard (or Streamlit app) with
#    ipywidgets sliders for n_estimators (1-500) and max_features
#    (1-8). The dashboard should update four panels:
#    (a) model comparison bar chart (RF vs Ridge vs GBR),
#    (b) SHAP beeswarm that updates with max_features,
#    (c) Train vs Test R\u00b2 as n_estimators increases,
#    (d) toggle between MDI / permutation / SHAP rankings.
#
# [Iterate] Use plotly.graph_objects, ipywidgets, shap, numpy,
# sklearn. Use the same variable names: X_train, X_test,
# y_train, y_test, data.feature_names. Do not use deprecated
# Plotly or SHAP functions.
#
# [Mechanism Check] Add inline comments explaining:
#   - How TreeExplainer differs from KernelExplainer
#   - Why SHAP values are additive (Shapley property)
#   - How ipywidgets observers trigger plot updates
#   - Why we re-fit inside the callback
#
# [Evaluate] Explain what the dashboard reveals about:
#   - The relationship between n_estimators, max_features,
#     and test performance
#   - Where MDI and SHAP rankings diverge and why
#   - The marginal value of additional trees beyond ~200

# PASTE AI-GENERATED CODE BELOW:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

# ============================================================
# DATA
# ============================================================
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Baseline models (fit once)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_r2 = r2_score(y_test, ridge.predict(X_test))

gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE
)
gbr.fit(X_train, y_train)
gbr_r2 = r2_score(y_test, gbr.predict(X_test))

# ============================================================
# SHAP UTILS (the three functions)
# ============================================================
def explain_prediction(model, X: pd.DataFrame, idx: int = 0, sample_size: int = 200):
    """SHAP waterfall for a single observation."""
    X_sample = X.sample(min(sample_size, len(X)), random_state=42).reset_index(drop=True)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample.values, approximate=True)
    explanation = shap.Explanation(
        values=sv,
        base_values=explainer.expected_value,
        data=X_sample.values,
        feature_names=X.columns.tolist(),
    )
    shap.plots.waterfall(explanation[idx])

def global_importance(model, X: pd.DataFrame, sample_size: int = 200):
    """SHAP beeswarm for global feature importance."""
    X_sample = X.sample(min(sample_size, len(X)), random_state=42).reset_index(drop=True)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample.values, approximate=True)
    explanation = shap.Explanation(
        values=sv,
        base_values=explainer.expected_value,
        data=X_sample.values,
        feature_names=X.columns.tolist(),
    )
    shap.plots.beeswarm(explanation)

def compare_importance(model, X: pd.DataFrame, y, sample_size: int = 200):
    """MDI vs SHAP vs Permutation side by side."""
    X_sample = X.sample(min(sample_size, len(X)), random_state=42).reset_index(drop=True)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample.values, approximate=True)

    mdi  = pd.Series(model.feature_importances_, index=X.columns)
    shap_imp = pd.Series(np.abs(sv).mean(axis=0), index=X.columns)
    perm = permutation_importance(model, X, y, n_repeats=5, random_state=42, n_jobs=-1)
    perm_imp = pd.Series(perm.importances_mean, index=X.columns)

    df = pd.DataFrame({"MDI": mdi, "SHAP": shap_imp, "Perm": perm_imp}).sort_values("SHAP", ascending=False)
    print(df.round(4).to_string())

    x = np.arange(len(df))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - 0.25, df["MDI"],  0.25, label="MDI",  color="#4C72B0")
    ax.bar(x,        df["SHAP"], 0.25, label="SHAP", color="#DD8452")
    ax.bar(x + 0.25, df["Perm"], 0.25, label="Perm", color="#55A868")
    ax.set_xticks(x)
    ax.set_xticklabels(df.index, rotation=35, ha="right")
    ax.legend()
    ax.set_title("MDI vs SHAP vs Permutation Importance")
    plt.tight_layout()
    plt.show()
    return df

# ============================================================
# DASHBOARD
# ============================================================
slider_n  = widgets.IntSlider(value=100, min=1,  max=500, step=1,  description="n_estimators", style={"description_width":"120px"}, layout=widgets.Layout(width="450px"))
slider_mf = widgets.IntSlider(value=3,   min=1,  max=8,   step=1,  description="max_features",  style={"description_width":"120px"}, layout=widgets.Layout(width="450px"))
toggle    = widgets.ToggleButtons(options=["MDI", "Permutation", "SHAP"], description="Panel (d):")
out       = widgets.Output()

def update(change=None):
    n_est = slider_n.value
    mf    = slider_mf.value

    # Re-fit RF with current slider values
    rf = RandomForestRegressor(n_estimators=n_est, max_features=mf, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_r2       = r2_score(y_test, rf.predict(X_test))
    rf_train_r2 = r2_score(y_train, rf.predict(X_train))

    # SHAP on sample
    X_sample = X_test.sample(150, random_state=42).reset_index(drop=True)
    explainer = shap.TreeExplainer(rf)
    sv = explainer.shap_values(X_sample.values, approximate=True)
    mean_shap = np.abs(sv).mean(axis=0)

    # R² curve
    n_grid = [1, 10, 25, 50, 100, 150, 200, 300, 500]
    train_r2s, test_r2s = [], []
    for n in n_grid:
        _rf = RandomForestRegressor(n_estimators=n, max_features=mf, random_state=RANDOM_STATE, n_jobs=-1)
        _rf.fit(X_train, y_train)
        train_r2s.append(r2_score(y_train, _rf.predict(X_train)))
        test_r2s.append(r2_score(y_test,  _rf.predict(X_test)))

    # Panel (d) importance
    imp_method = toggle.value
    if imp_method == "MDI":
        imp = pd.Series(rf.feature_importances_, index=data.feature_names)
    elif imp_method == "Permutation":
        perm = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
        imp  = pd.Series(perm.importances_mean, index=data.feature_names)
    else:
        imp = pd.Series(mean_shap, index=data.feature_names)

    # Build 4-panel Plotly figure
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f"(a) Model R² — n={n_est}, mf={mf}",
            "(b) SHAP Global Importance",
            "(c) Train vs Test R² curve",
            f"(d) Importance: {imp_method}",
        ],
        vertical_spacing=0.18,
    )

    # (a) bar chart
    fig.add_trace(go.Bar(
        x=["Ridge", f"RF (n={n_est})", "GBR"],
        y=[ridge_r2, rf_r2, gbr_r2],
        marker_color=["#5B8DB8", "#E8834A", "#55A868"],
        text=[f"{v:.4f}" for v in [ridge_r2, rf_r2, gbr_r2]],
        textposition="outside", showlegend=False,
    ), row=1, col=1)
    fig.update_yaxes(range=[0, 1.05], row=1, col=1)

    # (b) SHAP beeswarm proxy
    order = np.argsort(mean_shap)
    fig.add_trace(go.Bar(
        x=mean_shap[order],
        y=np.array(data.feature_names)[order],
        orientation="h",
        marker=dict(color=X_sample.values.mean(axis=0)[order], colorscale="RdBu_r", showscale=False),
        showlegend=False,
    ), row=1, col=2)

    # (c) R² curve
    fig.add_trace(go.Scatter(x=n_grid, y=train_r2s, name="Train R²", line=dict(color="#E8834A")), row=2, col=1)
    fig.add_trace(go.Scatter(x=n_grid, y=test_r2s,  name="Test R²",  line=dict(color="#5B8DB8")), row=2, col=1)
    fig.add_vline(x=n_est, line_dash="dash", line_color="gray", row=2, col=1)

    # (d) importance toggle
    imp_sorted = imp.sort_values()
    fig.add_trace(go.Bar(
        x=imp_sorted.values, y=imp_sorted.index.tolist(),
        orientation="h", marker_color="#9B59B6", showlegend=False,
    ), row=2, col=2)

    fig.update_layout(height=700, title_text=f"California Housing Explorer — n={n_est}, mf={mf}")

    with out:
        clear_output(wait=True)
        fig.show()
        print(f"\nRidge R²: {ridge_r2:.4f} | RF R²: {rf_r2:.4f} | GBR R²: {gbr_r2:.4f}")
        print(f"Train-test gap: {rf_train_r2 - rf_r2:.4f}")

slider_n.observe(update,  names="value")
slider_mf.observe(update, names="value")
toggle.observe(update,    names="value")

display(widgets.VBox([slider_n, slider_mf, toggle]), out)
update()  # initial render

# ============================================================
# SHAP UTILS — call these separately in the next cell
# ============================================================
# rf_100 = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
# global_importance(rf_100, X_test)
# explain_prediction(rf_100, X_test, idx=0)
# compare_importance(rf_100, X_test, y_test)

Output()

---
## Digital Portfolio: Institutional Signaling

### Generate Your Professional README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — README generation (no code, just docs)
# -----------------------------------------------------------

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Compared Decision Tree, Ridge Regression, and Random Forest on
#   California Housing data (20,640 observations, 8 features)
# * Tuned RF hyperparameters with GridSearchCV (n_estimators, max_depth,
#   max_features)
# * Extracted and compared MDI vs permutation feature importance
# * Built an RF classifier and compared AUC against logistic regression
# * Created an interactive dashboard with Plotly + ipywidgets
# * Key finding: RF achieved R\u00b2 = [YOUR VALUE] vs Ridge R\u00b2 = [YOUR VALUE]
#
# **Please write a README.md entry including:**
# 1. Project Title: Tree-Based Models \u2014 Random Forests
# 2. Objective: A professional one-sentence summary
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Make this sound like a professional tech economist wrote it."

### Tree-Based Models — Random Forests
**Objective:**
Benchmarked ensemble tree methods against linear baselines on the California Housing dataset to evaluate predictive performance, diagnose feature attribution bias, and deliver an interactive model exploration tool.

**Methodology:**

Compared Decision Tree, Ridge Regression, and Random Forest regressors across 20,640 observations and 8 engineered features, with correct train/test evaluation protocols
Tuned Random Forest hyperparameters (n_estimators, max_depth, max_features) via GridSearchCV with 5-fold cross-validation
Diagnosed MDI feature importance bias toward high-cardinality features; validated rankings against permutation importance on held-out data
Extended the pipeline to a classification task, benchmarking RF against Logistic Regression on AUC
Built an interactive four-panel Plotly dashboard with ipywidgets sliders exposing the relationship between hyperparameters, train/test R², and feature attribution in real time

**Key Findings:**

Random Forest (tuned) achieved R² = 0.8147 vs Ridge R² = 0.5759, demonstrating the performance ceiling of linear methods on spatially structured housing data
MDI and SHAP/permutation rankings diverged meaningfully for geographic features (Longitude, Latitude), confirming cardinality bias as a practical concern in production feature selection
Marginal R² gains from additional trees plateaued beyond ~150–200 estimators, establishing a cost-performance threshold relevant for deployment decisions
Gradient Boosting matched or exceeded tuned RF performance, consistent with its bias-reduction mechanism outperforming variance-averaging on this dataset

### Push to GitHub

```bash
cd econ-lab-19-random-forests
git add notebooks/ figures/ README.md verification-log.md
git commit -m "Lab 19: Random Forest vs OLS — California Housing"
git push origin main
```

Submit your GitHub repo link on Canvas.